In [1]:
#this is a generic code for looking at the continous HOBO data against the manual field measurements. Replace all of the "placeholders" with the files/parameters that you are looking for.

In [2]:
#first is a generic code that uses Temperature. After, I include the script for looking at SPC data of the same continuous and manual dataframes, but the beginning will also apply to SPC or another parameter.

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [4]:
 #install the package that runs a hampel filter--this is what cleans the spikes out of the data caused by removing the hobo sensors from the wells

In [5]:
pip install hampel

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
from hampel import hampel #this allows the hampel package to run

In [7]:
import os #allows python to communicate directly with the computer's operating system

In [8]:
#reading the continuous data
hobodf = pd.read_csv("P1_current.csv") #replace "hobodf" with a name for your location (e.g. "P1hobodf"). Replace "P1_current.csv" with the csv with the continuous data for your location.
hobodf["Date-Time"] = pd.to_datetime(hobodf["Date-Time"]) #This line converts the column named "Date-Time" to the pandas datetime format.
date = hobodf["Date-Time"] #replace "date" with whatever you want to name your date variable (e.g. "P1date"). Remember to replace "hobodf" with whatever you named the continuous dataset at the top of this cell.
sliceddate = date[1:] #In the code chunk below, I remove the first point from the parameter we are interested in (explained below). In order to plot this, I have to make the series the same length. This line removes the first date in the hobo data in order to make the pandas series the same length as the sliced temp data.

In [9]:
#filtering the continuous data for the outlier spikes where the hobo probe was removed from the piezometer
filteredhobo = hampel(hobodf["Temperature"], window_size=6, n_sigma=1.5) #filtering the data: for every point in the column "Temperature", a window of six data points on either side is applied, and the point is comparted to the median absolute deviation (with 1.5 standard deviations). If the data falls outside the MAD, it is counted as an outlier and removed. Replace "filteredhobo" with whatever you want to name the dataset.
filteredhoboseries = pd.Series(filteredhobo.filtered_data, index = hobodf.index) #the hampel filter changes the type of data, and this forces it back into a panda series.  
filteredhobo = filteredhoboseries[1:] #the hampel filter cannot remove the first data point (which I think is padded by zeroes), so I manually sliced it off. I'm overwriting "filteredhobo" to be the continuous data without the spikes, forced back into a series, with the first data point removed. 


KeyError: 'Temperature'

In [ ]:
#reading the manual data
mandf = pd.read_csv("Well_Sampling_Data_P1.csv", index_col=False) #replace "mandf" with a name for your location (e.g. "P1mandf").
mandf.loc[:,"Temp bottom before"] #selects the column in the manual data csv that you want on the y-axis. You can run the dataframe name ("mandf") to see what the columns are called.
mandf["Date"] = pd.to_datetime(mandf["Date"]) #converts the column called "Date" values to pandas datetime format. 


In [ ]:
#reading in the NOAA air temp data
weatherdf = pd.read_csv("NOAAHastingsDam2.csv") # replace "NOAAHastingsDam2.csv" with the name of the csv with your weather data.
weatherdf["Date"] = pd.to_datetime(weatherdf["Date"]) #converts "Date" column to pandas datetime format.


In [ ]:
#creating the plot
mandf.columns = mandf.columns.str.strip()  #This line was generated by Google Gemini to remove the leading and trailing whitespaces in the mandf dataframe.  

fig, ax = plt.subplots(layout='constrained')
ax.plot(sliceddate, filteredhobo, color="blue", label = "HOBO continuous temperature") #plots the filtered continuous data
ax.plot(mandf.Date, mandf.loc[:,"Temp top before"], 'x', color = "orange", label = "manual temperature at top before pumping") #each dataset in the manual field measurmemts data set is called by their column instead of naming each variable. Replace "mandf.Date" with nameofdataframe.nameofcolumnfortime and mandf.loc[:,"Temp top before"] with nameofdataframe.loc[:, "Name of column for parameter"]. 
ax.plot(mandf.Date, mandf.loc[:,"Temp bottom before"], 'x', color = "purple", label = "manual temperature at bottom before pumping")
ax.plot(mandf.Date, mandf.loc[:,"Temp top after"], 'o', color = "orange", label = "manual temperature at top after pumping")
ax.plot(mandf.Date, mandf.loc[:,"Temp bottom after"], 'o', color = "purple", label = "manual temperature at bottom after pumping")
ax.scatter(weatherdf.Date, weatherdf.loc[:,"Mean  Celcius Teperature"], color = "green", s = 1, label = "air temperature") #the air temperature data
plt.xticks(rotation=45) #rotate the x ticks so they are legible
plt.xlabel("Year-Month")
plt.ylabel("Temperature (°C)") #Temperature (°C) is a placeholder.


#some options for legends:
#plt.legend(bbox_to_anchor=(1.05, 0.5), loc="center left") #places legend outside graph, but squishes it
#ax.legend(loc="lower left") #places legend on graph and covers some of the data
fig.legend(loc='outside upper right') #puts legend outside of graph, squishes it down, but is not awful

plt.show()